In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy.random as rand

from math import pi, sqrt

In [2]:
# GLOBALS
NUM_PARTICLES = 100 # total number of particles
NUM_SWEEPS = 100000 # total number of Monte Carlo sweeps
DEL_R = 0.5 # Historgram width for counting number of particles 
SIGMA = 1.0 # Particle diameter

# random number generator
RNG = rand.default_rng()

In [3]:
# Parameters to be tuned/changed according to the problem
# volume fraction phi = pi * N * (sig)^3 / 6*V, but sig = 1 (particle diam)
# For 2-D case it's area fraction, phi = N * pi * (sig)^2 / (4 * L^2)
vol_frac = 0.45 

# Length of the entire box
# L = cube_root((N * pi) / (6*phi)) * sig
# For 2-D cases, the square box side is sqrt( (N * pi * sig^2)/ (4*phi))
box_side = (SIGMA / 2.)*sqrt((NUM_PARTICLES*pi)/vol_frac)
print(f"To achive area fraction of {vol_frac}, the side length is {box_side:.2f}")

To achive area fraction of 0.45, the side length is 13.21


In [4]:
def pbc(coord, box_side : float):
    """Check the periodic boundary condition of a particle, suppose we place a cubic box
    with side L with its center at the origin (0., 0., 0.). i.e. -L/2 <= r_a < L/2 for 
    each dimension a in {x, y, z}.
    Return r_a = r_a - L*[r_a/L], where [x] returns the nearest integer to x (could be 
    larger or smaller than x depending on the value of x)
    
    Parameters
    ----------
    coords : array
        (x, y, z) coord of the particle of interest.
    box_side : float
        The length of the entire box

    Returns
    -------
    pbc_coord : array
        New coordinates of the particle satisfied the periodic boundary condition
    """

    pbc_coord = coord - box_side*np.round(coord*(1/box_side))
    return pbc_coord

In [ ]:
# Demonstrating a 2-D example with L = 1.0
# Here the point's x-coord is inside the box, but the y-coord is outside the unit cell
test_coords = np.array([0.7, 0.6])
pbc_coords = pbc(test_coords, 1.0)

fig, ax = plt.subplots()
box = patches.Rectangle((-0.5, -0.5), width=1.0, height=1.0, edgecolor='k',
                        facecolor='None', linewidth=1.0)
ax.add_patch(box)
ax.scatter(test_coords[0], test_coords[1], label='Original')
ax.scatter(pbc_coords[0], pbc_coords[1], label='PBC')

In [5]:
def check_ovlp(cur_coords, prev_coords, dia=1.0):
    num_prev_particles = prev_coords.shape[0]
    for idx in np.arange(num_prev_particles):
        dist = np.linalg.norm(cur_coords - prev_coords[idx, :])
        if dist <= dia:
            # Found particles that overlap
            return True
        
    return False

In [6]:
# Initializing the ensemble
def init_ensemble(num_particles : int, box_side : float, dim : int, dia : float):
	ensemble = np.zeros((num_particles, dim))
	for idx in np.arange(num_particles):
		coord = RNG.uniform(size=dim, low=-box_side/2.0, high=box_side/2.0)
		# For the second and subsequent particles, we need to make sure we're
		# overlapping with existing particles in the box:
		if idx >= 1:
			while (check_ovlp(coord, prev_coords=ensemble[:idx, :], dia=dia) == True):
				coord = RNG.uniform(size=dim, low=-box_side/2.0, high=box_side/2.0)
		ensemble[idx, :] = coord

	return ensemble

In [7]:
def move_point(cur_coords, random_move, box_side : float, box_dist_len=1.0):
    '''Make a move
    '''

    new_coords = cur_coords + (2.0*random_move - 1.0)*box_dist_len

    return pbc(new_coords, box_side)

In [8]:
# Initializing
ensemble = init_ensemble(num_particles=NUM_PARTICLES,
						 box_side=box_side,
						 dim=2,
						 dia=SIGMA)

In [13]:
ensemble_copy = np.copy(ensemble)
num_accepted_moves = 0
for idx in np.arange(NUM_PARTICLES):
	# Move one particle at a time
	mask = np.ones(NUM_PARTICLES, dtype=bool)
	mask[idx] = False

	rand_move = RNG.uniform(low=0.0, high=1.0, size=2)
	cur_coords = ensemble_copy[idx, :]
	new_coords = move_point(cur_coords=cur_coords, random_move=rand_move,
						 box_side=box_side, box_dist_len=0.6)
	ovlp_cond = check_ovlp(new_coords, prev_coords=ensemble_copy[mask, :],
						dia=SIGMA)
	if ovlp_cond == False:
		num_accepted_moves += 1
		ensemble_copy[idx, :] = new_coords
print(f"Acceptance Ratio = {num_accepted_moves/NUM_PARTICLES:.2f}")

Acceptance Ratio = 0.38


In [14]:
MEASURE_INTERVAL = 10 # Measuring the G(r) every 10 MC sweeps
EQUIL_START = 10000

In [15]:
def compute_pairwise_dist(ensemble, box_side : float):
	num_particles = ensemble.shape[0]
	dim = ensemble.shape[1]

	pairwise_dist = np.zeros(int(num_particles*(num_particles+1)/2))
	
	cur_idx = 0
	for i_idx in np.arange(num_particles):
		if i_idx <= num_particles - 1:
			for j_idx in np.arange(i_idx+1, num_particles):
				dist_ij = np.linalg.norm(ensemble[i_idx, :] - ensemble[j_idx, :])
				if dist_ij > box_side/2:
					diff_vec_ij = ensemble[i_idx] - ensemble[j_idx]
					for dim_idx in np.arange(dim):
						if diff_vec_ij[dim_idx] > box_side/2:
							# Move it to the adjacent unit cell
							diff_vec_ij[dim_idx] -= box_side
						dist_ij = np.linalg.norm(diff_vec_ij)
				pairwise_dist[cur_idx] = dist_ij
				cur_idx += 1
	return pairwise_dist	

In [ ]:
# Run the simulation

NUM_BINS = 50
NUM_MEAS_ITER = (NUM_SWEEPS - EQUIL_START)/MEASURE_INTERVAL + 1

dist_histogram = []

for sweep_idx in np.arange(NUM_SWEEPS):
	for idx in np.arange(NUM_PARTICLES):
		# Move one particle at a time
		mask = np.ones(NUM_PARTICLES, dtype=bool)
		mask[idx] = False

		rand_move = RNG.uniform(low=0.0, high=1.0, size=2)
		cur_coords = ensemble[idx, :]
		new_coords = move_point(cur_coords=cur_coords, random_move=rand_move,
							box_side=box_side, box_dist_len=0.6)
		ovlp_cond = check_ovlp(new_coords, prev_coords=ensemble_copy[mask, :],
							dia=SIGMA)
		if ovlp_cond == False:
			ensemble[idx, :] = new_coords
	
	if sweep_idx + 1 >= EQUIL_START:
		if (sweep_idx + 1 % MEASURE_INTERVAL == 0):
			print("Measuring!")
			# Get the Histogram
			pairwise_dist = compute_pairwise_dist(ensemble=ensemble, box_side=box_side)
			hist = np.histogram(pairwise_dist, bins=NUM_BINS, range=(0.0, box_side/2))
			dist_histogram.append(hist)

